# NETOPT-EAST: Cost-Service Frontier & Correlation Sensitivity

This notebook evaluates:
1. **The Cost-Service Frontier:** How network costs trade off against customer fill rate $\alpha \in \{90\%, 95\%, 98\%, 99\%\}$.
2. **Spatial Correlation Sensitivity:** How distance-dependent demand correlation $\rho_0 \in \{0.0, 0.3, 0.6, 1.0\}$ erodes the textbook square-root pooling benefit.

In [ ]:
import sys, os
import pandas as pd
import plotly.io as pio

# Ensure project root is in path
sys.path.insert(0, os.path.abspath('..'))

from src.optimization.frontier import frontier_insights
from src.visualization.cost_charts import plot_cost_service_frontier, plot_corrosion_of_pooling

# Load processed frontier and sensitivity datasets
frontier_df = pd.read_csv('../data/processed/frontier_results.csv')
sens_df = pd.read_csv('../data/processed/correlation_sensitivity.csv')
insights = frontier_insights(frontier_df)

## 1. Cost-Service Frontier Results

Below is the full cost breakdown as the service level targets increase from 90% to 99%.

In [ ]:
display_cols = ['service_level', 'total_cost', 'fixed_cost', 'transport_cost', 'inventory_cost', 'safety_stock_cost', 'n_warehouses']
styled_frontier = frontier_df[display_cols].copy()
styled_frontier['service_level'] = styled_frontier['service_level'].map(lambda x: f'{x:.0%}')
for col in ['total_cost', 'fixed_cost', 'transport_cost', 'inventory_cost', 'safety_stock_cost']:
    styled_frontier[col] = styled_frontier[col].map(lambda x: f'₹{x:,.0f}')

styled_frontier

### **Headline Insight**

> **Moving from 95% to 99% fill rate costs ₹9 lakh/year (+0.2% total network cost).**

## 2. Interactive Frontier Chart (Stacked Cost Composition)

In [ ]:
fig_frontier = plot_cost_service_frontier(
    frontier_df,
    output_html='../reports/figures/frontier.html',
    output_png='../reports/figures/frontier.png'
)
fig_frontier.show()

## 3. Correlation Sensitivity: Corrosion of the Square-Root Law

Evaluating the impact of spatial correlation $\rho_0$ on the safety stock pooling benefit.

In [ ]:
display_sens = sens_df.copy()
display_sens['safety_stock_cost'] = display_sens['safety_stock_cost'].map(lambda x: f'₹{x:,.0f}')
display_sens['total_cost'] = display_sens['total_cost'].map(lambda x: f'₹{x:,.0f}')
display_sens['pooling_benefit_pct'] = display_sens['pooling_benefit_pct'].map(lambda x: f'{x:.1f}%')
display_sens[['rho_0', 'safety_stock_cost', 'pooling_benefit_pct', 'open_warehouses']]

## 4. Interactive Corrosion of Pooling Chart

In [ ]:
fig_corrosion = plot_corrosion_of_pooling(
    sens_df,
    output_html='../reports/figures/corrosion.html',
    output_png='../reports/figures/corrosion.png'
)
fig_corrosion.show()

## 5. Strategic Insights for Supply Chain Leadership

> **Key Executive Takeaway:**
>
> At a 95% fill rate, the optimal FMCG distribution network costs **₹51.09 Crore/year**. Pushing customer fill rate to 99% requires a **41.4% increase in safety stock units**, yet increases total annual expenditure by only **₹9.4 Lakh/year (+0.18%)** because fixed warehouse leases and primary/secondary freight dominate total logistics spend.
>
> However, spatial correlation analysis reveals that the textbook square-root law is an optimistic upper bound. When demand is completely independent ($\rho_0 = 0$), pooling yields a **65.8% reduction** in safety stock compared to decentralized node holding. Under high local spatial correlation ($\rho_0 = 1.0$), that benefit decreases to **50.7%**. In geographically dispersed networks across Eastern India, substantial pooling benefit survives even under high local correlation because distant demand clusters (e.g., Guwahati vs. Kolkata) remain statistically independent due to spatial decay.